# Descarga y exportación de imágenes desde Google Earth Engine

Este notebook documenta la generación de un stack multibanda Sentinel-2 para el flujo de clasificación de coberturas en humedales de Bogotá.

El proceso incluye:

- autenticación e inicialización de Google Earth Engine;
- carga del ROI local;
- filtrado de imágenes Sentinel-2;
- enmascaramiento de nubes;
- cálculo de índices espectrales;
- cálculo de texturas GLCM;
- visualización interactiva del stack;
- exportación del resultado.

La exportación puede realizarse hacia Google Drive, Earth Engine Assets o Google Cloud Storage, según la configuración del usuario.

In [1]:
from pathlib import Path
import sys

import ee

In [2]:
BASE_DIR = Path.cwd()

while not (BASE_DIR / "src").exists() and BASE_DIR != BASE_DIR.parent:
    BASE_DIR = BASE_DIR.parent

sys.path.append(str(BASE_DIR))

print("BASE_DIR detectado:")
print(BASE_DIR)

BASE_DIR detectado:
C:\Users\AVDON\JupyterLab\ATENEA\humedales_bogota_ml


## Importación de funciones auxiliares

Las funciones principales para trabajar con Google Earth Engine se encuentran en:

`src/clasificacion_coberturas/gee.py`

Este módulo contiene la lógica para cargar el ROI, crear el stack Sentinel-2, visualizarlo y exportarlo.

In [3]:
from src.clasificacion_coberturas.gee import (
    inicializar_gee,
    cargar_roi_como_ee,
    crear_stack_sentinel2,
    visualizar_stack_gee,
    exportar_stack,
    obtener_lista_bandas,
    guardar_lista_bandas
)

## Autenticación e inicialización de Google Earth Engine

La autenticación debe realizarse con una cuenta que tenga acceso a Google Earth Engine.

El parámetro `GEE_PROJECT` debe modificarse según el proyecto de Earth Engine del usuario o entidad que ejecute el flujo.

In [4]:
# ============================================================
# AUTENTICACIÓN E INICIALIZACIÓN
# ============================================================

GEE_PROJECT = None

ee.Authenticate(
    force=True,
    scopes=[
        "https://www.googleapis.com/auth/earthengine",
        "https://www.googleapis.com/auth/drive"
    ]
)

inicializar_gee(project=GEE_PROJECT)

Enter verification code:  4/1AeoWuM_bA_uDRzoDM3UrGqMJ49J-yRE4eQjC4b8jX9varl8WFaaPA1wsblA



Successfully saved authorization token.
Google Earth Engine inicializado correctamente.


## Configuración de rutas

Se definen las rutas locales del proyecto.

El ROI debe estar almacenado en la carpeta de datos crudos del flujo de clasificación de coberturas.

Se recomienda evitar tildes, espacios y caracteres especiales en nombres de archivos.

In [5]:
# ============================================================
# RUTAS
# ============================================================

ruta_roi = (
    BASE_DIR
    / "data"
    / "raw"
    / "clasificacion_coberturas"
    / "vectores"
    / "ROI_BOGOTA.shp"
)

carpeta_rasters_local = (
    BASE_DIR
    / "data"
    / "raw"
    / "clasificacion_coberturas"
    / "rasters"
)

carpeta_tablas = (
    BASE_DIR
    / "outputs"
    / "tables"
    / "clasificacion_coberturas"
)

print("Ruta ROI:")
print(ruta_roi)

print("\nCarpeta local sugerida para rasters descargados:")
print(carpeta_rasters_local)

print("\nCarpeta de tablas:")
print(carpeta_tablas)

Ruta ROI:
C:\Users\AVDON\JupyterLab\ATENEA\humedales_bogota_ml\data\raw\clasificacion_coberturas\vectores\ROI_BOGOTA.shp

Carpeta local sugerida para rasters descargados:
C:\Users\AVDON\JupyterLab\ATENEA\humedales_bogota_ml\data\raw\clasificacion_coberturas\rasters

Carpeta de tablas:
C:\Users\AVDON\JupyterLab\ATENEA\humedales_bogota_ml\outputs\tables\clasificacion_coberturas


## Configuración del stack Sentinel-2

Se definen los parámetros principales para construir el stack:

- año de análisis;
- porcentaje máximo de nubosidad;
- escala espacial de exportación;
- tamaño de ventana para texturas GLCM;
- bandas utilizadas para el cálculo de texturas.

El flujo se deja configurado con el año 2018 para mantener compatibilidad con los archivos ya generados localmente. Para generar otro año, por ejemplo 2024, se debe modificar `year` y el nombre de salida.

In [6]:
# ============================================================
# PARÁMETROS DEL STACK
# ============================================================

year = 2018
cloud_pct = 15
scale = 10
glcm_size = 3

tex_bands = [
    "B2",
    "B3",
    "B4",
    "B8"
]

nombre_salida = "S2_2018_STACK_3"

ruta_bandas_csv = (
    carpeta_tablas
    / f"bandas_stack_{nombre_salida}.csv"
)

print("Año:", year)
print("Nubosidad máxima:", cloud_pct)
print("Escala de exportación:", scale)
print("Tamaño GLCM:", glcm_size)
print("Bandas para texturas:", tex_bands)
print("Nombre de salida:", nombre_salida)

Año: 2018
Nubosidad máxima: 15
Escala de exportación: 10
Tamaño GLCM: 3
Bandas para texturas: ['B2', 'B3', 'B4', 'B8']
Nombre de salida: S2_2018_STACK_3


## Configuración de exportación

Google Earth Engine no exporta imágenes grandes directamente al disco local de forma estándar.

Por esta razón, se dejan disponibles tres modos de exportación:

- `drive`: exporta a Google Drive;
- `asset`: exporta a un Asset de Earth Engine;
- `cloud_storage`: exporta a Google Cloud Storage.

Para este flujo se usa `drive` por defecto, ya que replica la lógica original y es práctico para usuarios técnicos. En flujos institucionales o empresariales, `cloud_storage` puede ser más robusto si existe un bucket configurado.

In [7]:
# ============================================================
# CONFIGURACIÓN DE EXPORTACIÓN
# ============================================================

modo_exportacion = "drive"  # opciones: "drive", "asset", "cloud_storage"

drive_folder = "GEE_Coberturas_Humedales"

asset_id = None
# Ejemplo:
# asset_id = "projects/mi-proyecto/assets/S2_2018_STACK3"

cloud_bucket = None
# Ejemplo:
# cloud_bucket = "mi-bucket-gee"

descripcion_tarea = nombre_salida
file_name_prefix = nombre_salida

print("Modo de exportación:")
print(modo_exportacion)

print("\nCarpeta Drive:")
print(drive_folder)

print("\nAsset ID:")
print(asset_id)

print("\nCloud bucket:")
print(cloud_bucket)

Modo de exportación:
drive

Carpeta Drive:
GEE_Coberturas_Humedales

Asset ID:
None

Cloud bucket:
None


## Carga del ROI

Se carga el ROI local y se convierte a un objeto de Earth Engine.

El ROI se reproyecta a EPSG:4326 antes de enviarlo a GEE.

In [8]:
gdf_roi, roi_ee, region = cargar_roi_como_ee(ruta_roi)

print("ROI cargado correctamente.")
print("Número de geometrías:", len(gdf_roi))
print("CRS local:", gdf_roi.crs)

ROI cargado correctamente.
Número de geometrías: 1
CRS local: EPSG:4326


## Generación del stack Sentinel-2

Se construye un stack multibanda a partir de Sentinel-2 SR Harmonized.

El stack incluye:

- bandas espectrales base;
- índices espectrales;
- texturas GLCM calculadas sobre bandas seleccionadas.

In [9]:
stack_final = crear_stack_sentinel2(
    region=region,
    year=year,
    cloud_pct=cloud_pct,
    glcm_size=glcm_size,
    tex_bands=tex_bands
)

print("Stack Sentinel-2 generado correctamente.")

Stack Sentinel-2 generado correctamente.


## Lista de bandas del stack

Se obtiene y guarda la lista de bandas generadas en el stack.

Este archivo permite documentar el orden y nombre de las variables que componen el raster exportado.

In [10]:
bandas_stack = obtener_lista_bandas(stack_final)

df_bandas = guardar_lista_bandas(
    bandas=bandas_stack,
    ruta_salida=ruta_bandas_csv
)

display(df_bandas)

Lista de bandas guardada en:
C:\Users\AVDON\JupyterLab\ATENEA\humedales_bogota_ml\outputs\tables\clasificacion_coberturas\bandas_stack_S2_2018_STACK_3.csv


,orden,banda
0,1,B2
1,2,B3
2,3,B4
3,4,B8
4,5,B5
5,6,B6
6,7,B7
7,8,B8A
8,9,B11
9,10,B12


## Visualización interactiva

Se visualiza el stack generado sobre el ROI mediante `geemap`.

Esta visualización permite verificar que la imagen cubra correctamente el área de estudio antes de iniciar la exportación.

In [11]:
mapa = visualizar_stack_gee(
    stack=stack_final,
    roi_ee=roi_ee,
    region=region,
    year=year,
    vis_bands=["B4", "B3", "B2"],
    vis_min=0,
    vis_max=3000,
    zoom=12
)

mapa

Map(center=[4.643600831346751, -74.10830489791186], controls=(WidgetControl(options=['position', 'transparent_…

## Exportación del stack

Se inicia la exportación del stack según el modo seleccionado.

La opción por defecto es Google Drive. Una vez finalizada la tarea en GEE, el usuario debe descargar el archivo GeoTIFF y ubicarlo en la carpeta local correspondiente:

`data/raw/clasificacion_coberturas/rasters/`

In [12]:
task = exportar_stack(
    stack=stack_final,
    region=region,
    modo_exportacion=modo_exportacion,
    description=descripcion_tarea,
    scale=scale,
    drive_folder=drive_folder,
    file_name_prefix=file_name_prefix,
    asset_id=asset_id,
    cloud_bucket=cloud_bucket
)

Exportación a Google Drive iniciada.
Descripción de tarea: S2_2018_STACK_3
Carpeta en Drive: GEE_Coberturas_Humedales
Nombre de archivo: S2_2018_STACK_3


## Verificación de la tarea

La exportación se ejecuta como una tarea asincrónica en Google Earth Engine.

El estado puede consultarse desde la interfaz web de Earth Engine o mediante el objeto `task`.

In [13]:
print("Estado inicial de la tarea:")
print(task.status())

Estado inicial de la tarea:
{'state': 'RUNNING', 'description': 'S2_2018_STACK_3', 'priority': 100, 'creation_timestamp_ms': 1777761589687, 'update_timestamp_ms': 1777761699472, 'start_timestamp_ms': 1777761593449, 'task_type': 'EXPORT_IMAGE', 'attempt': 1, 'id': 'OXMMXOAJRUQMQMWJGJ5IK3JI', 'name': 'projects/36269736218/operations/OXMMXOAJRUQMQMWJGJ5IK3JI'}


## Producto de esta etapa

Al finalizar este notebook, se espera contar con un archivo GeoTIFF exportado desde Google Earth Engine.

Para este flujo, el archivo esperado es:

`S2_2018_STACK3.tif`

Después de finalizar la tarea de exportación en Google Earth Engine, el archivo debe descargarse y ubicarse localmente en:

`data/raw/clasificacion_coberturas/rasters/`

Este raster constituye el insumo principal para la siguiente etapa del flujo.

## Siguiente etapa del flujo

Una vez descargado y ubicado localmente el stack multibanda, el siguiente paso consiste en recortar o separar los raster por humedal.

Esta fase se documenta en:

`02_recorte_humedales.ipynb`